# pivotai — Phase 4: Generate Model Responses

Runs inference for all 3 pivotai GGUFs against the 92-record golden set.  
Output: `responses_<timestamp>.jsonl` saved to Google Drive.  
Estimated time on T4: **~1 hour** (vs ~23 hrs on MacBook Air CPU).

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Add `HF_TOKEN` to Colab Secrets (left sidebar 🔑) if your HF repos are private
3. Upload `golden_set.jsonl` when Cell 2 prompts you

In [ ]:
# Cell 1 — Install dependencies
# llama-cpp-python with CUDA backend for T4 GPU inference
!pip install -q huggingface_hub tqdm
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

# Verify GPU is visible
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'NOT FOUND — check runtime type')

In [ ]:
# Cell 2 — Upload golden_set.jsonl
from google.colab import files
import json, pathlib

print('Upload golden_set.jsonl from data/evals/ on your Mac...')
uploaded = files.upload()   # triggers file picker

golden_path = pathlib.Path('golden_set.jsonl')
if 'golden_set.jsonl' not in uploaded:
    raise FileNotFoundError('Expected golden_set.jsonl — please re-upload')
golden_path.write_bytes(uploaded['golden_set.jsonl'])

golden_records = [json.loads(l) for l in golden_path.read_text().splitlines() if l.strip()]
print(f'Loaded {len(golden_records)} golden records')

In [ ]:
# Cell 3 — Mount Google Drive (output survives session end)
from google.colab import drive
drive.mount('/content/drive')

import pathlib
OUTPUT_DIR = pathlib.Path('/content/drive/MyDrive/pivotai_evals')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# Cell 4 — Download GGUFs from HuggingFace
# If your repos are private: add HF_TOKEN to Colab Secrets (left sidebar 🔑)
# then uncomment the login line below.

from huggingface_hub import hf_hub_download, login
from google.colab import userdata
import pathlib, os

# Uncomment if repos are private:
# login(token=userdata.get('HF_TOKEN'))

HF_USER = 'ishreyadev'
MODELS_DIR = pathlib.Path('/content/models')
MODELS_DIR.mkdir(exist_ok=True)

GGUF_FILES = {
    'pivotai-ft':         (f'{HF_USER}/pivotai-ft-gguf',         'pivotai_ft.gguf'),
    'pivotai-distill':    (f'{HF_USER}/pivotai-distill-gguf',    'pivotai_distill.gguf'),
    'pivotai-curriculum': (f'{HF_USER}/pivotai-curriculum-gguf', 'pivotai_curriculum.gguf'),
}

model_paths = {}
for model_name, (repo_id, filename) in GGUF_FILES.items():
    dest = MODELS_DIR / filename
    if dest.exists():
        print(f'{model_name}: already downloaded ({dest.stat().st_size / 1e9:.1f} GB)')
    else:
        print(f'Downloading {model_name} from {repo_id}...')
        hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(MODELS_DIR))
        print(f'  → {dest.stat().st_size / 1e9:.1f} GB')
    model_paths[model_name] = str(dest)

print('\nAll models ready:', list(model_paths.keys()))

In [ ]:
# Cell 5 — Inference helpers (mirrors phase4_evals/utils.py exactly)
import json

SLM_FT_MODEL         = 'pivotai-ft'
SLM_DIST_MODEL       = 'pivotai-distill'
SLM_CURRICULUM_MODEL = 'pivotai-curriculum'

FT_INSTRUCTION = (
    'Act as pivotai Optimizer. Given a traveler persona for an Indian domestic trip, '
    'produce an optimized day-by-day itinerary that minimizes total cost while respecting '
    'the budget tier, trip type, and traveler intents. Identify the primary Price-Pivot Point '
    '(transit, accommodation, or activity substitution that saves ≥5%) and explain it clearly.'
)

DISTILL_INSTRUCTION = (
    'Act as pivotai Supervisor for an Indian domestic trip. Coordinate the Analyst, '
    'Concierge, and Optimizer agents to find Price-Pivot Points and produce an optimized '
    'itinerary. Show the reasoning chain for each agent handoff, then provide the final '
    'pivot analysis and optimized itinerary.'
)

# distill needs more tokens — outputs reasoning chain before JSON
NUM_PREDICT = {
    SLM_DIST_MODEL:       1024,
    SLM_FT_MODEL:          512,
    SLM_CURRICULUM_MODEL:  512,
}

def build_prompt(persona: dict, model_name: str) -> str:
    instruction = DISTILL_INSTRUCTION if model_name == SLM_DIST_MODEL else FT_INSTRUCTION
    persona_str = json.dumps(persona, ensure_ascii=False)
    return f'### Instruction:\n{instruction}\n\n### Input:\n{persona_str}\n\n### Response:\n'

print('Helpers defined')

In [ ]:
# Cell 6 — Run inference for all 3 models
# Loads each GGUF onto the T4 GPU, generates responses, saves JSONL to Drive.
# Resume-safe: skips (model_name, record_id) pairs already in the output file.

import json, pathlib, time
from datetime import datetime, timezone
from llama_cpp import Llama
from tqdm.notebook import tqdm

ALL_MODELS = [SLM_FT_MODEL, SLM_DIST_MODEL, SLM_CURRICULUM_MODEL]

timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
output_path = OUTPUT_DIR / f'responses_{timestamp}.jsonl'
print(f'Output: {output_path}')

# Resume: collect already-done (model_name, record_id) pairs from any existing file
done_keys = set()
for existing in OUTPUT_DIR.glob('responses_*.jsonl'):
    for line in existing.read_text().splitlines():
        try:
            r = json.loads(line)
            if not r.get('raw_output', '').startswith('ERROR:'):
                done_keys.add((r['model_name'], r['golden_record_id']))
        except Exception:
            pass
print(f'Already done: {len(done_keys)} records (will skip)')

total_success = total_error = 0

for model_name in ALL_MODELS:
    gguf_path = model_paths[model_name]
    n_predict = NUM_PREDICT[model_name]

    print(f'\n── Loading {model_name} onto GPU...')
    t0 = time.time()
    llm = Llama(
        model_path=gguf_path,
        n_gpu_layers=-1,   # offload all layers to T4
        n_ctx=2048,
        verbose=False,
    )
    print(f'   Loaded in {time.time()-t0:.1f}s')

    model_success = model_error = 0
    for golden in tqdm(golden_records, desc=model_name):
        record_id = golden.get('id') or golden.get('record_id', '')
        if not record_id:
            record_id = json.dumps(golden.get('persona', {}), sort_keys=True)[:32]

        if (model_name, record_id) in done_keys:
            continue

        persona = golden.get('persona', {})
        prompt  = build_prompt(persona, model_name)

        try:
            out = llm(
                prompt,
                max_tokens=n_predict,
                temperature=0.3,
                top_p=0.9,
                echo=False,
            )
            raw = out['choices'][0]['text']
            model_success += 1
        except Exception as exc:
            raw = f'ERROR: {exc}'
            model_error += 1

        record = {
            'model_name':       model_name,
            'golden_record_id': record_id,
            'persona':          persona,
            'prompt_used':      prompt,
            'raw_output':       raw,
            'generated_at':     datetime.now(timezone.utc).isoformat(),
        }
        with open(output_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f'   {model_name}: {model_success} ok, {model_error} errors')
    total_success += model_success
    total_error   += model_error

    # Free GPU memory before loading next model
    del llm

print(f'\n✓ Done: {total_success} generated, {total_error} errors')
print(f'  Output: {output_path}')
print('\nDownload this file and place it in data/evals/ on your Mac.')
print('Then run: python phase4_evals/score_responses.py')

In [ ]:
# Cell 7 — Download responses file directly to your Mac (optional)
# Alternatively just grab it from Google Drive.
from google.colab import files
files.download(str(output_path))